In [49]:
import svm_xgb_ensemble as model
import importlib
from svm_xgb_ensemble import tune_svm
importlib.reload(model)
print(model.SEED)

1027


In [50]:
categorical_cols = ["merchant_category","card_type","auth_method","channel","device_type","is_foreign_transaction","is_new_merchant","used_vpn","ip_country_mismatch","billing_shipping_mismatch","is_ai_generated_scam_attempt"]

In [51]:
df = model.pd.read_csv('data/train.csv')
y = df["is_fraud"]
# Drop ID and target columns from the raw data.
raw = df.drop(columns=["is_fraud", "id"])

# Splits categorical and numerical data.
if categorical_cols is not None:
    categorical = categorical_cols
    unknown = sorted(set(categorical) - set(raw.columns))
    if unknown:
        raise ValueError(f"Categorical column(s) not present: {unknown}")
else:
    categorical = raw.select_dtypes(exclude=["number"]).columns.tolist()
numerical = [c for c in raw.columns if c not in categorical]

if not numerical:
    raise ValueError("No numerical features found: check categorical column selection")
if len(numerical) != 13 or len(categorical) != 11:
    print(f"WARNING: detected {len(numerical)} numerical and {len(categorical)} "
            "categorical features, rather than the expected 13/11. "
            "Use --categorical-cols if category codes are numeric.")

x = model.prepare_features(raw, categorical=categorical, numerical=numerical)



In [52]:
# 70% train, 15% calibration, 15% untouched test.
x_train, x_rest, y_train, y_rest = model.train_test_split(
x, y, test_size=0.30, stratify=y, random_state=model.SEED)
x_cal, x_test, y_cal, y_test = model.train_test_split(
x_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=model.SEED)

In [53]:
ensemble = model.make_ensemble(numerical=numerical, categorical=categorical, components=1000)
ensemble.fit(x_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('svm', ...), ('xgb', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",Pipeline(step...state=1027))])
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",3
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'predict_proba'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,) or list of ndarray if `y` is of type `""multilabel-indicator""`.Class labels.","ndarray[float64](2,)","[0.,1.]"
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[CalibratedCla...od='isotonic'), Pipeline(step...=None, ...))])]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[object](

In [54]:
print("Calibrating final ensemble on independent calibration split...")
calibration_inputs = model.to_logit(ensemble.predict_proba(x_cal)[:, 1])
calibrator = model.LogisticRegression(max_iter=1000, random_state=model.SEED)
calibrator.fit(calibration_inputs, y_cal)

Calibrating final ensemble on independent calibration split...


,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",1027
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [57]:
import joblib
joblib.dump({
        "ensemble": ensemble,
        "calibrator": calibrator,
        # For downstream prediction only; not selected or evaluated by ROC AUC.
        "threshold": 0.5,
        "numerical": numerical,
        "categorical": categorical,
    }, "fraud_model.joblib")
print(f"Saved model to: fraud_model.joblib")

Saved model to: fraud_model.joblib


In [ ]:
import importlib
import svm_xgb_ensemble

importlib.reload(svm_xgb_ensemble)

from svm_xgb_ensemble import tune_svm

from svm_xgb_ensemble import tune_svm
best_svm = tune_svm(x_train, y_train, numerical)

# Evaluate on untouched test data
svm_scores = best_svm.decision_function(x_test)

print(
    "Test ROC AUC:",
    model.roc_auc_score(y_test, svm_scores)
)

In [35]:
import joblib
saved = joblib.load("fraud_model.joblib")
ensemble = saved["ensemble"]
calibrator = saved["calibrator"]
numerical = saved["numerical"]
categorical = saved["categorical"]

In [55]:
# ROC AUC is threshold-independent: use probabilities, not binary predictions.
# The positive class (1) represents fraudulent transactions.
p_test = model.calibrated_predict(ensemble, calibrator, x_test)
p_svm = ensemble.named_estimators_["svm"].predict_proba(x_test)[:, 1]
p_xgb = ensemble.named_estimators_["xgb"].predict_proba(x_test)[:, 1]

In [56]:
print("\n=== Untouched test set: ROC AUC (primary metric) ===")
print(f"Fraud ensemble ROC AUC: {model.roc_auc_score(y_test, p_test):.5f}")
print(f"SVM-only ROC AUC:       {model.roc_auc_score(y_test, p_svm):.5f}")
print(f"XGBoost-only ROC AUC:   {model.roc_auc_score(y_test, p_xgb):.5f}")
print("Note: ROC AUC does not depend on a classification threshold.")


=== Untouched test set: ROC AUC (primary metric) ===
Fraud ensemble ROC AUC: 0.94219
SVM-only ROC AUC:       0.88179
XGBoost-only ROC AUC:   0.94481
Note: ROC AUC does not depend on a classification threshold.


In [ ]:
import csv

test_df = model.pd.read_csv('data/test.csv')
raw = test_df.drop(columns=["id"])

# Splits categorical and numerical data.
if categorical_cols is not None:
    categorical = categorical_cols
    unknown = sorted(set(categorical) - set(raw.columns))
    if unknown:
        raise ValueError(f"Categorical column(s) not present: {unknown}")
else:
    categorical = raw.select_dtypes(exclude=["number"]).columns.tolist()
numerical = [c for c in raw.columns if c not in categorical]

if not numerical:
    raise ValueError("No numerical features found: check categorical column selection")
if len(numerical) != 13 or len(categorical) != 11:
    print(f"WARNING: detected {len(numerical)} numerical and {len(categorical)} "
            "categorical features, rather than the expected 13/11. "
            "Use --categorical-cols if category codes are numeric.")

x = model.prepare_features(raw, categorical=categorical, numerical=numerical)

predictions = model.calibrated_predict(ensemble, calibrator, x)
results_df = model.pd.DataFrame({
    'id': test_df['id'],
    'is_fraud': predictions
})
results_df.to_csv('results/results6.csv', index=False)